# Hyperparameter tuning (Optuna) — Colab tier

The local models use hand-fixed LightGBM params for every market. This notebook
searches per-market params with Optuna under the same `TimeSeriesSplit` CV the
trainer uses, then writes the tuned params back for the laptop to train with.

Flow:
1. Local: `python run.py --export-features --sport nba` (and nhl / tennis)
2. Upload the parquet(s) to Drive `sports-edge/exports/`
3. Run this on Colab (CPU is fine), then download `*_tuned_params.json` into
   `data/exports/artifacts/` and `python run.py --import-models` picks them up.


In [ ]:
!pip -q install optuna lightgbm polars scikit-learn pyarrow


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
BASE='/content/drive/MyDrive/sports-edge'; EXPORTS=BASE+'/exports'; ARTIFACTS=BASE+'/artifacts'
os.makedirs(ARTIFACTS, exist_ok=True)


In [ ]:
# Tune one market with Optuna under TimeSeriesSplit (mirrors models.train._cv_score).
import json, numpy as np, polars as pl, optuna, lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

def cv_logloss(X, y, params):
    tscv=TimeSeriesSplit(n_splits=5); sc=[]
    for tr,va in tscv.split(X):
        d=lgb.Dataset(X[tr],label=y[tr]); v=lgb.Dataset(X[va],label=y[va],reference=d)
        m=lgb.train(params,d,num_boost_round=400,valid_sets=[v],
                    callbacks=[lgb.early_stopping(30,verbose=False),lgb.log_evaluation(0)])
        p=np.clip(m.predict(X[va]),1e-7,1-1e-7)
        sc.append(-np.mean(y[va]*np.log(p)+(1-y[va])*np.log(1-p)))
    return float(np.mean(sc))

def objective_factory(X,y):
    def obj(t):
        params={'objective':'binary','metric':'binary_logloss','verbose':-1,
            'learning_rate':t.suggest_float('learning_rate',0.01,0.1,log=True),
            'num_leaves':t.suggest_int('num_leaves',15,127),
            'min_data_in_leaf':t.suggest_int('min_data_in_leaf',10,100),
            'feature_fraction':t.suggest_float('feature_fraction',0.5,1.0),
            'bagging_fraction':t.suggest_float('bagging_fraction',0.5,1.0),
            'bagging_freq':t.suggest_int('bagging_freq',1,10),
            'lambda_l1':t.suggest_float('lambda_l1',1e-3,10,log=True),
            'lambda_l2':t.suggest_float('lambda_l2',1e-3,10,log=True)}
        return cv_logloss(X,y,params)
    return obj


In [ ]:
# Example: tune NBA moneyline. Repeat per market/target by swapping the parquet+target.
import sys; sys.path.append('/content/sports-edge')  # clone the repo here first
from models.train import TEAM_FEATURE_COLS
df=pl.read_parquet(EXPORTS+'/nba_game_features.parquet').filter(pl.col('is_home')==1)
cols=[c for c in TEAM_FEATURE_COLS if c in df.columns]
sub=df.select(cols+['win']).drop_nulls(subset=['win']).fill_nan(None)
X=sub.select(cols).to_numpy(); y=sub['win'].to_numpy()
study=optuna.create_study(direction='minimize')
study.optimize(objective_factory(X,y), n_trials=40, show_progress_bar=True)
print('best logloss',study.best_value); print(study.best_params)
json.dump({'nba_moneyline':study.best_params}, open(ARTIFACTS+'/nba_tuned_params.json','w'), indent=2)
print('wrote', ARTIFACTS+'/nba_tuned_params.json')


Plug the tuned params into `models/train.py` (or load the JSON there) and
retrain locally. Tune each market the same way by changing the target column
(`win` -> `margin`/`total`/prop stat) and objective (binary vs regression).
